# Final Algorithm Comparison

This notebook compares the final MDGP local-search heuristic with the selected comparison algorithms in terms of solution quality, winner rate, and run time.

In [10]:
from pathlib import Path

import numpy as np
import pandas as pd

## Configuration

In [11]:
GRAPH_ORDER = [
    "powerlaw",
    "er",
]

DATASET_ORDER = [
    "small sparse",
    "small dense",
    "large sparse",
    "large dense",
]

ALGORITHM_ORDER = [
    "leiden",
    "leiden_mdgp",
    "kapoce",
    "mdgp_plateau",
]

RESULTS_DIR = Path("../../results/experiment3")
RAW_RESULTS_FILE = RESULTS_DIR / "raw_results.csv"

## Load data

In [12]:
raw = pd.read_csv(RAW_RESULTS_FILE)

raw["dataset_group"] = raw["size_class"].astype(str) + " " + raw["regime"].astype(str)

raw["graph_type"] = pd.Categorical(
    raw["graph_type"],
    categories=GRAPH_ORDER,
    ordered=True,
)

raw["dataset_group"] = pd.Categorical(
    raw["dataset_group"],
    categories=DATASET_ORDER,
    ordered=True,
)

raw["algorithm"] = pd.Categorical(
    raw["algorithm"],
    categories=ALGORITHM_ORDER,
    ordered=True,
)

## Relative solution quality

For every instance, the best solution found by any of the compared algorithms is used as the reference.

Relative solution quality is defined as

$
\frac{\text{best solution quality on the instance}}
{\text{solution quality of the respective algorithm}}.
$

A value of $1.0$ indicates that the algorithm attains the best solution found on the instance. Values greater than $1.0$ indicate the remaining quality gap.

In [13]:
instance_keys = [
    "graph_type",
    "dataset_group",
    "dataset",
    "instance",
]

density_table = raw.pivot(index=instance_keys, columns="algorithm", values="density",)[ALGORITHM_ORDER]

best_per_instance = density_table.max(axis=1)
relative_to_best = density_table.rdiv(best_per_instance, axis=0)

relative_to_best.columns.name = "algorithm"

quality_summary = (
    relative_to_best
    .groupby(level=["graph_type", "dataset_group"])
    .mean()
    .stack()
    .rename("mean_relative_to_best")
    .reset_index()
)

quality_summary

,graph_type,dataset_group,algorithm,mean_relative_to_best
0,powerlaw,small sparse,leiden,2.970392
1,powerlaw,small sparse,leiden_mdgp,1.095257
2,powerlaw,small sparse,kapoce,1.049708
3,powerlaw,small sparse,mdgp_plateau,1.000000
4,powerlaw,small dense,leiden,2.538343
5,powerlaw,small dense,leiden_mdgp,1.132796
6,powerlaw,small dense,kapoce,1.047734
7,powerlaw,small dense,mdgp_plateau,1.000000
8,powerlaw,large sparse,leiden,9.873555
9,powerlaw,large sparse,leiden_mdgp,1.098397


## Winner rate

An algorithm is counted as a winner whenever it attains the best solution found on an instance. Ties are counted for all involved algorithms.

In [14]:
winner_summary = (
    density_table
    .eq(best_per_instance, axis=0)
    .groupby(level=["graph_type", "dataset_group"])
    .mean()
    .stack()
    .rename("winner_rate")
    .reset_index()
)

winner_summary

,graph_type,dataset_group,algorithm,winner_rate
0,powerlaw,small sparse,leiden,0.000
1,powerlaw,small sparse,leiden_mdgp,0.000
2,powerlaw,small sparse,kapoce,0.000
3,powerlaw,small sparse,mdgp_plateau,1.000
4,powerlaw,small dense,leiden,0.000
5,powerlaw,small dense,leiden_mdgp,0.000
6,powerlaw,small dense,kapoce,0.000
7,powerlaw,small dense,mdgp_plateau,1.000
8,powerlaw,large sparse,leiden,0.000
9,powerlaw,large sparse,leiden_mdgp,0.000


## Run time

The arithmetic mean of the complete run time of each algorithm is computed over all instances in the corresponding dataset group.

In [15]:
runtime_summary = (
    raw
    .groupby(["graph_type", "dataset_group", "algorithm"], observed=True, as_index=False)
    .agg(mean_runtime=("runtime", "mean"))
)

runtime_summary

,graph_type,dataset_group,algorithm,mean_runtime
0,powerlaw,small sparse,leiden,0.053121
1,powerlaw,small sparse,leiden_mdgp,0.004071
2,powerlaw,small sparse,kapoce,0.142645
3,powerlaw,small sparse,mdgp_plateau,30.379303
4,powerlaw,small dense,leiden,0.015139
5,powerlaw,small dense,leiden_mdgp,0.003337
6,powerlaw,small dense,kapoce,0.041539
7,powerlaw,small dense,mdgp_plateau,52.823186
8,powerlaw,large sparse,leiden,0.107400
9,powerlaw,large sparse,leiden_mdgp,0.015293


In [16]:
comparison_summary = (
    quality_summary
    .merge(winner_summary, on=["graph_type", "dataset_group", "algorithm"])
    .merge(runtime_summary, on=["graph_type", "dataset_group", "algorithm"])
)

comparison_summary["algorithm"] = pd.Categorical(
    comparison_summary["algorithm"],
    categories=ALGORITHM_ORDER,
    ordered=True,
)

comparison_summary = (
    comparison_summary
    .sort_values(["graph_type", "dataset_group", "algorithm"])
    .reset_index(drop=True)
)

comparison_summary

,graph_type,dataset_group,algorithm,mean_relative_to_best,winner_rate,mean_runtime
0,powerlaw,small sparse,leiden,2.970392,0.000,0.053121
1,powerlaw,small sparse,leiden_mdgp,1.095257,0.000,0.004071
2,powerlaw,small sparse,kapoce,1.049708,0.000,0.142645
3,powerlaw,small sparse,mdgp_plateau,1.000000,1.000,30.379303
4,powerlaw,small dense,leiden,2.538343,0.000,0.015139
5,powerlaw,small dense,leiden_mdgp,1.132796,0.000,0.003337
6,powerlaw,small dense,kapoce,1.047734,0.000,0.041539
7,powerlaw,small dense,mdgp_plateau,1.000000,1.000,52.823186
8,powerlaw,large sparse,leiden,9.873555,0.000,0.107400
9,powerlaw,large sparse,leiden_mdgp,1.098397,0.000,0.015293


## LaTeX helper functions

In [17]:
def truncate_number(value: float, decimals: int) -> float:
    factor = 10 ** decimals
    return np.trunc(value * factor) / factor


def latex_algorithm(algorithm: str) -> str:
    return r"\texttt{" + algorithm.replace("_", r"\_") + "}"


def format_number(value: float, decimals: int) -> str:
    return f"{truncate_number(value, decimals):.{decimals}f}"


def format_percent(value: float, decimals: int) -> str:
    return f"{truncate_number(100 * value, decimals):.{decimals}f}" + r"\,\%"

## Build LaTeX tables

In [20]:
def make_comparison_latex_table(df: pd.DataFrame, caption: str, label: str) -> str:
    graph_labels = {
        "powerlaw": "Powerlaw",
        "er": "Erdős-Rényi",
    }

    lines = [
        r"\begin{table}[!htbp]",
        r"\centering",
        rf"\caption{{{caption}}}",
        rf"\label{{{label}}}",
        r"\begin{tabular}{p{2.0cm}p{1.9cm}p{2.2cm}rrr}",
        r"\toprule",
        r"Graph type & Dataset & Algorithm & \shortstack{Mean relative\\solution quality} & Win rate & \shortstack{Mean\\run time (s)} \\",
        r"\midrule",
    ]

    for graph_index, graph_type in enumerate(GRAPH_ORDER):
        graph_df = df[df["graph_type"] == graph_type]

        graph_row_count = len(graph_df)
        current_graph_row = 0

        for dataset_index, dataset in enumerate(DATASET_ORDER):
            part = graph_df[graph_df["dataset_group"] == dataset].sort_values("algorithm")

            best_quality = part["mean_relative_to_best"].min()

            for row_index, row in enumerate(part.itertuples(index=False)):
                graph_cell = (
                    rf"\multirow{{{graph_row_count}}}{{*}}{{{graph_labels[graph_type]}}}"
                    if current_graph_row == 0
                    else ""
                )

                dataset_cell = (
                    rf"\multirow{{{len(part)}}}{{*}}{{{dataset}}}"
                    if row_index == 0
                    else ""
                )

                quality = format_number(row.mean_relative_to_best, 6)

                if np.isclose(row.mean_relative_to_best, best_quality):
                    quality = rf"\textbf{{{quality}}}"

                lines.append(
                    f"{graph_cell} "
                    f"& {dataset_cell} "
                    f"& {latex_algorithm(str(row.algorithm))} "
                    f"& {quality} "
                    f"& {format_percent(row.winner_rate, 1)} "
                    f"& {format_number(row.mean_runtime, 3)} "
                    r"\\"
                )

                current_graph_row += 1

            if dataset_index < len(DATASET_ORDER) - 1:
                lines.append(r"\cmidrule(l){2-6}")

        if graph_index < len(GRAPH_ORDER) - 1:
            lines.append(r"\midrule")

    lines.extend(
        [
            r"\bottomrule",
            r"\end{tabular}",
            r"\end{table}",
        ]
    )

    return "\n".join(lines)

In [21]:
comparison_latex = make_comparison_latex_table(
    comparison_summary,
    caption=(
        "Mean relative solution quality, win rate, and mean run time of the compared algorithms."
    ),
    label="tab:algorithm_comparison",
)

print(comparison_latex)

\begin{table}[!htbp]
\centering
\caption{Mean relative solution quality, win rate, and mean run time of the compared algorithms.}
\label{tab:algorithm_comparison}
\begin{tabular}{p{2.0cm}p{1.9cm}p{2.2cm}rrr}
\toprule
Graph type & Dataset & Algorithm & \shortstack{Mean relative\\solution quality} & Win rate & \shortstack{Mean\\run time (s)} \\
\midrule
\multirow{16}{*}{Powerlaw} & \multirow{4}{*}{small sparse} & \texttt{leiden} & 2.970392 & 0.0\,\% & 0.053 \\
 &  & \texttt{leiden\_mdgp} & 1.095257 & 0.0\,\% & 0.004 \\
 &  & \texttt{kapoce} & 1.049707 & 0.0\,\% & 0.142 \\
 &  & \texttt{mdgp\_plateau} & \textbf{1.000000} & 100.0\,\% & 30.379 \\
\cmidrule(l){2-6}
 & \multirow{4}{*}{small dense} & \texttt{leiden} & 2.538343 & 0.0\,\% & 0.015 \\
 &  & \texttt{leiden\_mdgp} & 1.132795 & 0.0\,\% & 0.003 \\
 &  & \texttt{kapoce} & 1.047733 & 0.0\,\% & 0.041 \\
 &  & \texttt{mdgp\_plateau} & \textbf{1.000000} & 100.0\,\% & 52.823 \\
\cmidrule(l){2-6}
 & \multirow{4}{*}{large sparse} & \texttt{le